1. Импорты и установка зависимостей (если нужно)

In [8]:
# !pip install sentence-transformers pdfplumber python-docx PyYAML tqdm ipywidgets pandas
# При необходимости раскомментируйте строку выше и выполните её один раз

from __future__ import annotations

import os
from pathlib import Path
from typing import List, Dict, Any, Tuple

import json
import yaml
import csv

import pdfplumber
from docx import Document

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer, util

from tqdm.notebook import tqdm
from ipywidgets import VBox, HBox, HTML

2. Вспомогательные функции загрузки данных

In [9]:
# --- Загрузка текста спецификации ---
def load_text_from_file(file_path: str) -> str:
    """Загружает текст из .txt, .pdf или .docx файла."""
    path = Path(file_path)
    ext = path.suffix.lower()

    if ext == ".txt":
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    elif ext == ".pdf":
        texts = []
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text() or ""
                texts.append(page_text)
        return "\n".join(texts)

    elif ext == ".docx":
        doc = Document(str(path))
        texts = [p.text for p in doc.paragraphs]
        return "\n".join(texts)

    else:
        raise ValueError(f"Неподдерживаемый формат файла спецификации: {ext}")


# --- Загрузка критериев ---
def load_criteria_from_file(file_path: str) -> List[Dict[str, Any]]:
    """Загружает критерии из .json, .yaml/.yml или .csv.

    Ожидается, что в каждом критерии есть поля: name, description, importance
    (importance – число, например от 1 до 5).
    """
    path = Path(file_path)
    ext = path.suffix.lower()

    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = data.get("criteria", [])
        return list(data)

    elif ext in {".yaml", ".yml"}:
        with open(path, "r", encoding="utf-8") as f:
            data = yaml.safe_load(f)
        if isinstance(data, dict):
            data = data.get("criteria", [])
        return list(data)

    elif ext == ".csv":
        df = pd.read_csv(path)
        return df.to_dict(orient="records")

    else:
        raise ValueError(f"Неподдерживаемый формат файла критериев: {ext}")


def normalize_criteria(raw_criteria: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Приводит критерии к стандартному виду: name, description, importance (float)."""
    normalized = []
    for c in raw_criteria:
        name = c.get("name") or c.get("Название") or c.get("title")
        description = c.get("description") or c.get("Описание")
        importance = c.get("importance") or c.get("Важность") or 1
        try:
            importance = float(importance)
        except Exception:
            importance = 1.0

        normalized.append({
            "name": name,
            "description": description,
            "importance": importance,
        })
    return normalized

3. Функции разбиения текста спецификации

In [10]:
def split_spec_into_chunks(
    text: str,
    mode: str = "paragraphs",
    min_length: int = 20
) -> List[str]:
    """Разбивает текст ТЗ на куски (абзацы или предложения).

    mode: "paragraphs" или "sentences".
    min_length: минимальная длина куска, короче – склеиваются с соседями.
    """
    text = text.replace("\r", "")

    chunks: List[str] = []

    if mode == "paragraphs":
        raw_parts = [p.strip() for p in text.split("\n\n") if p.strip()]
    elif mode == "sentences":
        # Простое разбиение по предложениям (можно заменить на spaCy/NLTK при желании)
        import re
        raw_parts = re.split(r"(?<=[.!?])\s+", text)
        raw_parts = [p.strip() for p in raw_parts if p.strip()]
    else:
        raise ValueError("mode должен быть 'paragraphs' или 'sentences'")

    buffer = []
    current_len = 0
    for part in raw_parts:
        buffer.append(part)
        current_len += len(part)
        if current_len >= min_length:
            chunks.append(" ".join(buffer).strip())
            buffer = []
            current_len = 0

    if buffer:
        chunks.append(" ".join(buffer).strip())

    return chunks

4. Функции эмбеддингов и оценки критериев

In [11]:
class SpecEvaluator:
    def __init__(
        self,
        model_name: str = "T-pro-it-2.0",
    ):
        """Инициализация клиента эмбеддингов поверх языковой модели T-pro-it-2.0.

        Предполагается, что у вас есть доступ к API/клиенту T-pro-it-2.0, который
        позволяет получать векторные представления (эмбеддинги) для списка
        текстов. Внутри этого класса вы можете адаптировать код под ваш
        конкретный способ вызова модели (HTTP API, SDK и т.п.).
        """
        self.model_name = model_name

        # Здесь вместо SentenceTransformer можно инициализировать ваш клиент T-pro-it-2.0.
        # Пример (псевдокод):
        # from your_client_lib import TProClient
        # self.client = TProClient(model=model_name, api_key=os.getenv("TPRO_API_KEY"))

    def embed_texts(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Создаёт эмбеддинги для списка текстов с прогресс-баром, используя T-pro-it-2.0.

        Адаптируйте тело функции под фактический интерфейс вашей модели
        T-pro-it-2.0. На выходе должен получаться numpy-массив формы
        (N, D), где N — количество текстов, D — размерность эмбеддинга.
        """
        embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
            batch = texts[i:i + batch_size]

            # Замените этот блок на реальный вызов T-pro-it-2.0.
            # Псевдокод:
            # vecs = self.client.embed(batch)  # Должен вернуть np.ndarray

            # Временно создадим заглушку нулевых векторов, чтобы структура кода была полной.
            if not embeddings:
                dim = 768  # замените на фактическую размерность эмбеддингов T-pro-it-2.0
            vecs = np.zeros((len(batch), dim), dtype="float32")

            embeddings.append(vecs)

        if not embeddings:
            return np.zeros((0, 0), dtype="float32")

        return np.vstack(embeddings)


def evaluate_criteria(
    spec_chunks: List[str],
    criteria: List[Dict[str, Any]],
    model: SpecEvaluator,
    similarity_threshold: float = 0.55,
    top_k_evidence: int = 3,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Оценивает каждый критерий по семантическому сходству с частями ТЗ."""

    if not spec_chunks:
        raise ValueError("Список spec_chunks пуст. Проверьте разбиение спецификации.")

    crit_descriptions = [c["description"] for c in criteria]

    # Эмбеддинги
    spec_embeds = model.embed_texts(spec_chunks)
    crit_embeds = model.embed_texts(crit_descriptions)

    # Матрица косинусных сходств: (num_criteria, num_chunks)
    similarities = util.cos_sim(crit_embeds, spec_embeds).cpu().numpy()

    results = []
    gaps = []
    strengths = []

    for i, crit in enumerate(criteria):
        sim_row = similarities[i]
        best_idx = int(np.argmax(sim_row))
        best_score = float(sim_row[best_idx])

        # Выбираем топ-k фрагментов как evidence
        top_indices = np.argsort(sim_row)[::-1][:top_k_evidence]
        evidence_spans = [
            {
                "index": int(idx),
                "similarity": float(sim_row[idx]),
                "text": spec_chunks[int(idx)][:500]  # ограничим длину выдержки
            }
            for idx in top_indices
        ]

        met = best_score >= similarity_threshold
        confidence = max(0.0, min(1.0, best_score))

        if met:
            explanation = (
                f"Критерий считается выполненным: максимальное семантическое сходство "
                f"между описанием критерия и фрагментом ТЗ = {best_score:.2f}, "
                f"что выше порога {similarity_threshold:.2f}."
            )
            strengths.append(crit["name"] or crit["description"])
        else:
            explanation = (
                f"Критерий НЕ считается выполненным: максимальное семантическое сходство "
                f"= {best_score:.2f}, что ниже порога {similarity_threshold:.2f}. "
                "Возможно, критерий описан неявно или отсутствует в ТЗ."
            )
            gaps.append(crit["name"] or crit["description"])

        results.append({
            "name": crit["name"],
            "met": "Да" if met else "Нет",
            "met_bool": met,
            "confidence": confidence,
            "best_similarity": best_score,
            "importance": crit["importance"],
            "evidence_spans": evidence_spans,
            "explanation": explanation,
        })

    df = pd.DataFrame(results)

    # Взвешенная общая оценка (0–100%)
    total_weight = df["importance"].sum()
    if total_weight > 0:
        weighted_score = (100.0 * (df["met_bool"] * df["importance"]).sum() / total_weight)
    else:
        weighted_score = 0.0

    summary = {
        "weighted_score": float(weighted_score),
        "gaps": gaps,
        "strengths": strengths,
    }

    return df, summary

5. Визуализация и отображение отчёта в Jupyter

In [12]:
def style_results_dataframe(df: pd.DataFrame) -> pd.io.formats.style.Styler:
    """Возвращает стилизованный DataFrame с подсветкой выполненных/невыполненных критериев."""

    def color_met(val):
        if val == "Да":
            return "background-color: #064e3b; color: #ecfdf5;"
        else:
            return "background-color: #7f1d1d; color: #fee2e2;"

    def color_confidence(val):
        # Градиент от красного к зелёному
        v = float(val)
        r = int(255 * (1 - v))
        g = int(255 * v)
        return f"background-color: rgb({r}, {g}, 80); color: #f9fafb;"

    display_cols = ["name", "met", "confidence", "importance", "best_similarity", "explanation"]
    df_disp = df[display_cols].copy()

    styler = (
        df_disp.style
        .applymap(color_met, subset=["met"])
        .applymap(color_confidence, subset=["confidence"])
        .format({
            "confidence": lambda x: f"{x:.2f}",
            "best_similarity": lambda x: f"{x:.2f}",
            "importance": lambda x: f"{x:g}",
        })
    )
    return styler


def display_report(df: pd.DataFrame, summary: Dict[str, Any]):
    """Отображает сводный отчёт в Jupyter Notebook."""
    weighted_score = summary.get("weighted_score", 0.0)
    gaps = summary.get("gaps", [])
    strengths = summary.get("strengths", [])

    score_color = "#b91c1c"
    if weighted_score >= 80:
        score_color = "#166534"
    elif weighted_score >= 60:
        score_color = "#15803d"
    elif weighted_score >= 40:
        score_color = "#ca8a04"

    header_html = HTML(
        f"""<div style='padding: 12px; border-radius: 8px; background:#020617; color:#e5e7eb; border:1px solid #1f2937;'>
        <div style='font-size:14px; text-transform:uppercase; letter-spacing:.08em; color:#9ca3af; margin-bottom:4px;'>
            Общая оценка соответствия
        </div>
        <div style='font-size:28px; font-weight:700; color:{score_color};'>
            {weighted_score:.1f}%
        </div>
        </div>"""
    )

    gaps_html = HTML(
        """<div style='margin-top:12px;'>
        <div style='font-weight:600; color:#f97316; margin-bottom:4px;'>Основные пробелы:</div>
        <ul style='margin-left:18px; color:#e5e7eb; font-size:13px;'>""" +
        "".join(
            [f"<li>{g}</li>" for g in gaps] or ["<li>Явных пробелов не обнаружено (по семантической оценке).</li>"]
        ) +
        """</ul></div>"""
    )

    strengths_html = HTML(
        """<div style='margin-top:12px;'>
        <div style='font-weight:600; color:#22c55e; margin-bottom:4px;'>Сильные стороны:</div>
        <ul style='margin-left:18px; color:#e5e7eb; font-size:13px;'>""" +
        "".join(
            [f"<li>{s}</li>" for s in strengths] or ["<li>Выраженных сильных сторон не выделено.</li>"]
        ) +
        """</ul></div>"""
    )

    display(VBox([header_html, HBox([gaps_html, strengths_html])]))

    display(style_results_dataframe(df))

6. Основная обёртка: удобный вызов оценки

In [13]:
def evaluate_specification(
    spec_source: str,
    criteria_source: Any,
    spec_is_text: bool = False,
    criteria_is_list: bool = False,
    split_mode: str = "paragraphs",
    min_chunk_len: int = 50,
    similarity_threshold: float = 0.55,
    model_name: str = "sentence-transformers/distiluse-base-multilingual-cased-v1",
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Высокоуровневая функция оценки ТЗ.

    spec_source: текст ТЗ или путь к файлу (.txt, .pdf, .docx)
    criteria_source: список критериев (list[dict]) или путь к файлу (.json, .yaml, .csv)
    """
    # 1. Получаем текст спецификации
    if spec_is_text:
        spec_text = spec_source
    else:
        spec_text = load_text_from_file(spec_source)

    # 2. Загружаем критерии
    if criteria_is_list:
        raw_criteria = criteria_source
    else:
        raw_criteria = load_criteria_from_file(criteria_source)

    criteria = normalize_criteria(raw_criteria)

    # 3. Разбиваем ТЗ на куски
    spec_chunks = split_spec_into_chunks(spec_text, mode=split_mode, min_length=min_chunk_len)

    # 4. Инициализируем модель
    evaluator = SpecEvaluator(model_name=model_name)

    # 5. Оцениваем критерии
    df, summary = evaluate_criteria(
        spec_chunks=spec_chunks,
        criteria=criteria,
        model=evaluator,
        similarity_threshold=similarity_threshold,
    )

    return df, summary

7. Пример использования в Jupyter Notebook

In [14]:
# Пример: оценка спецификации из PDF и критериев из JSON/YAML

spec_path = r'C:\Users\troyd\OneDrive\Desktop\Стажировка\ТЗ Электронный журнал_v2.1.docx'       # замените на путь к вашему ТЗ (.pdf/.docx/.txt)
criteria_path = r'C:\Users\troyd\OneDrive\Desktop\Стажировка\Критерии.csv'  # или .json / .csv


# Запуск оценки
df_results, summary = evaluate_specification(
    spec_source=spec_path,
    criteria_source=criteria_path,
    spec_is_text=False,
    criteria_is_list=False,
    split_mode="paragraphs",   # или "sentences"
    min_chunk_len=80,
    similarity_threshold=0.6,
)

# Отображение отчёта
display_report(df_results, summary)

# При желании можно также посмотреть сырые данные evidence_spans
# df_results.head()[["name", "evidence_spans"]]

Embedding:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\troyd\AppData\Local\Temp\ipykernel_7424\2151165948.py:22: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(color_met, subset=["met"])
C:\Users\troyd\AppData\Local\Temp\ipykernel_7424\2151165948.py:23: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(color_confidence, subset=["confidence"])


,name,met,confidence,importance,best_similarity,explanation
0,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
1,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
2,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
3,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
4,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
5,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
6,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
7,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
8,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
9,None,Нет,0.00,1,0.00,"Критерий НЕ считается выполненным: максимальное семантическое сходство = 0.00, что ниже порога 0.60. Возможно, критерий описан неявно или отсутствует в ТЗ."
